In [ ]:
import sys
import math
from enum import Enum

class Policy(Enum):
    FIRST_FIT = "first_fit"
    BEST_FIT = "best_fit"
    POOLING = "pooling"

print("=== Heap Diagnostic Code Loaded ===")

class Heap:
    def __init__(self, size, policy=Policy.FIRST_FIT):
        self.blocks = [(0, size, True)]  # (start, size, free)
        self.policy = policy

        if policy == Policy.BEST_FIT:
            self.malloc_func = self.malloc_best_fit
        elif policy == Policy.FIRST_FIT:
            self.malloc_func = self.malloc_first_fit
        elif policy == Policy.POOLING:
            self.malloc_func = self.malloc_pooling
            self.prealloc_pooling(size)
        else:
            raise ValueError("Invalid policy")

    def dump(self):
        print("Heap:", self.blocks)

    def prealloc_pooling(self, size):
        self.blocks.clear()
        if size < 16:
            raise ValueError("Size must be at least 16")
        start_pow = min(7, math.floor(math.log2(size)) - 1)
        alloc_counts = {}

        block_sum = sum(1 << i for i in range(start_pow, 3, -1))
        remaining = size
        for i in range(start_pow, 3, -1):
            block_size = 1 << i
            if i >= 6:
                budget = math.floor(((block_size) * size)/block_sum)
                blocks = budget // block_size
            else:
                blocks = remaining // block_size
            if blocks:
                remaining -= blocks * (block_size)
                alloc_counts[block_size] = blocks
        print(alloc_counts)

        start = 0
        for block_size in alloc_counts.keys():
            for j in range(alloc_counts[block_size]):
                self.blocks.append((start, block_size, True))
                start += block_size

    def malloc(self, size):
        print(f"\nCALL: malloc({size})")

        start = self.malloc_func(size)
        if start != -1:
            return start

        # ---- FORCED FAILURE PATH ----
        free_blocks = [bsize for _, bsize, free in self.blocks if free]
        total_free = sum(free_blocks)
        largest = max(free_blocks) if free_blocks else 0

        print(">>> ALLOCATION FAILED <<<")
        print("TOTAL FREE :", total_free)
        print("LARGEST BLK:", largest)
        print("REQUESTED :", size)

        if total_free >= size:
            raise RuntimeError("FRAGMENTATION ERROR DETECTED")
        else:
            raise RuntimeError("OUT OF MEMORY")

    def free(self, ptr):
        print(f"\nCALL: free({ptr})")
        for i, (start, size, free) in enumerate(self.blocks):
            if start == ptr and not free:
                self.blocks[i] = (start, size, True)
                print("FREED")
                return
        raise RuntimeError("INVALID FREE")

    def malloc_best_fit(self, size):
        index = -1
        diff = sys.maxsize
        for i, (start, bsize, free) in enumerate(self.blocks):
            if free and bsize >= size:
                temp_diff = bsize - size
                if(temp_diff < diff):
                    diff = temp_diff
                    index = i
        if index != -1:
            block = self.blocks[index]
            start, bsize, _ = block
            self.blocks[index] = (start, size, False)
            if bsize > size:
                self.blocks.insert(index + 1, (start + size, bsize - size, True))
            print("ALLOCATED")
            return start
        return -1

    def malloc_first_fit(self, size):
        for i, (start, bsize, free) in enumerate(self.blocks):
            if free and bsize >= size:
                self.blocks[i] = (start, size, False)
                if bsize > size:
                    self.blocks.insert(i + 1, (start + size, bsize - size, True))
                print("ALLOCATED")
                return start
        return -1

    def malloc_pooling(self, size):
        target = 0
        for i in range(4, 8):
            target = 1 << i
            if(target >= size):
                break
        if target < size:
            return -1

        while target <= 128:
            for i, (start, bsize, free) in enumerate(self.blocks):
                if free and bsize == target:
                    self.blocks[i] = (start, bsize, False)
                    print("ALLOCATED")
                    return start
            target <<= 1

        run_start = None
        run_size = 0
        run_indices = []

        for i, (start, bsize, free) in enumerate(self.blocks):
            if free:
                if run_start is None:
                    run_start = start
                run_size += bsize
                run_indices.append(i)

                if run_size >= size:
                    for idx in run_indices:
                        s, bs, _ = self.blocks[idx]
                        self.blocks[idx] = (s, bs, False)
                    return run_start
            else:
                run_start = None
                run_size = 0
                run_indices.clear()

        return -1

=== Heap Diagnostic Code Loaded ===


In [ ]:
heap = Heap(100, Policy.POOLING)

p1 = heap.malloc(30)
heap.dump()

p2 = heap.malloc(20)
heap.dump()

heap.free(p1)
heap.dump()

heap.free(p2)
heap.dump()

p3 = heap.malloc(20)
heap.dump()

{32: 3}

CALL: malloc(30)
ALLOCATED
Heap: [(0, 32, False), (32, 32, True), (64, 32, True)]

CALL: malloc(20)
ALLOCATED
Heap: [(0, 32, False), (32, 32, False), (64, 32, True)]

CALL: free(0)
FREED
Heap: [(0, 32, True), (32, 32, False), (64, 32, True)]

CALL: free(32)
FREED
Heap: [(0, 32, True), (32, 32, True), (64, 32, True)]

CALL: malloc(20)
ALLOCATED
Heap: [(0, 32, False), (32, 32, True), (64, 32, True)]


In [ ]:
heap = Heap(100, Policy.POOLING)

a = heap.malloc(20)   # 0–19
b = heap.malloc(20)   # 20–39
c = heap.malloc(20)   # 40–59
d = heap.malloc(20)   # 60–79
heap.dump()

heap.free(b)          # free 20–39
heap.free(d)          # free 60–79
heap.dump()

# Total free = 40
# Largest block = 20
heap.malloc(25)       # MUST FAIL (fragmentation)

{32: 3}

CALL: malloc(20)
ALLOCATED

CALL: malloc(20)
ALLOCATED

CALL: malloc(20)
ALLOCATED

CALL: malloc(20)
>>> ALLOCATION FAILED <<<
TOTAL FREE : 0
LARGEST BLK: 0
REQUESTED : 20


RuntimeError: OUT OF MEMORY

In [ ]:
heap = Heap(50, Policy.POOLING)
heap.dump()

p1 = heap.malloc(30)
p2 = heap.malloc(15)

heap.dump()

# Truly out of memory
p3 = heap.malloc(10)

{16: 3}
Heap: [(0, 16, True), (16, 16, True), (32, 16, True)]

CALL: malloc(30)

CALL: malloc(15)
ALLOCATED
Heap: [(0, 16, False), (16, 16, False), (32, 16, False)]

CALL: malloc(10)
>>> ALLOCATION FAILED <<<
TOTAL FREE : 0
LARGEST BLK: 0
REQUESTED : 10


RuntimeError: OUT OF MEMORY

In [ ]:
heap.free(999)   # Invalid pointer


CALL: free(999)


RuntimeError: INVALID FREE